# Show how to build a RAG app with a tutorial framework

In [1]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data

documents = load_faq_data()

In [4]:
from sqlitesearch import TextSearchIndex

# 1. Point to your existing db file with the exact same schema fields
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db",  # Adjust the path if the file is in a different folder
)

In [12]:
instructions = """
You're a course teaching assistant.

You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyse the results, and then perform more searches if needed. 

Try to expand your search by using new keywords based on the results you get from the search.


The question has to be about the course or its logistics, offtopic questions shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.

If you can't find an answer, return `I don't know.`
""".strip()

### Add the `search` function to the tool

In [ ]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
    )

In [8]:
agent_tools = Tools()
agent_tools.add_tool(
    search
)  #  adding a type hint & docstring to search function allow us to not need to declare the function schema (e.g. skip the JSON file)

In [9]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

### Add the chat interface and runner

In [16]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)


### Run a single query

In [17]:
result = runner.loop(
    prompt="how to run ollama locallys",
    callback=callback,
)

-> Response received


-> Response received


In [18]:
result.cost

CostInfo(input_cost=Decimal('0.00117'), output_cost=Decimal('0.0010845'), total_cost=Decimal('0.0022545'))

### multi-turn convos with convo history

In [ ]:
result2 = runner.loop(
    prompt="how do I run a diff model",
    previous_messages=result.all_messages,  # add convo history
    callback=callback,
)

-> Response received


### Interactive chat
* For a chat-like workflow, run the built-in input loop:
* Type questions and get answers. Type "stop" to exit.

In [ ]:
runner.run()